<h1 style="text-align: center;">Hotel Booking Cancellation Prediction Using Machine Learning</h1>
<h3 style="text-align: center;">Valencia & Muhammad Rafi Andrianto</h3>

---

## **Section 0. Setup**

### **0.1 Import Library**

In [23]:
# common library
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Feature Engineering
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from category_encoders import BinaryEncoder
from feature_engine.outliers import Winsorizer, OutlierTrimmer
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest
from sklearn.model_selection import train_test_split
from feature_engine.selection import DropFeatures
from feature_engine.datetime import DatetimeFeatures

# Model and pipelining
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Evaluation
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import cross_validate
from sklearn.model_selection import RandomizedSearchCV

# Imbalance
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

### **0.2 Global Configuration**

In [2]:
RANDOM_STATE = 42
pd.set_option('display.max_columns', None)

## **Section 1. Business Understanding**

**Background**

Hotel booking cancellations create uncertainty in occupancy planning and reduce potential revenue. This project analyzes hotel booking data from a hotel in Portugal to develop a machine learning model that predicts whether a customer will cancel their reservation. Early identification of booking cancellations enables hotels to reduce revenue loss, optimize room occupancy, and improve operational planning.

**Key Stakeholders**
- Hotel General Manager: responsible for hotel performance and business strategy.
- Revenue Management Team: responsible for hotel pricing, occupancy, and revenue.
- Front Office Team: responsible for room reservations, booking policies and coordinating guest arrivals and room allocation.
- Data Science Team: provides analytical insights and builds predictive tools to support business decisions.

**Problem Statement**

Hotel booking cancellations represent a significant operational and financial challenge for the hotel. With a substantial number of reservations being canceled before check-in, the hotel faces:
- **Revenue loss** from rooms that remain vacant because canceled bookings cannot always be replaced in time.
- **Operational challenges** in managing room allocation and staff planning..
- **Difficulty managing room availability and pricing**, reducing opportunities to maximize occupancy and revenue.
- **Difficulty forecasting future bookings**, making it harder to plan hotel operations and promotions.

Without accurately identifying customers who are likely to cancel, the hotel cannot proactively implement strategies such as deposit requirements, personalized reminders, or targeted retention offers to minimize cancellations and improve occupancy rates.

**Goals**

To address this challenge, the hotel aims to:
- **Analyze cancellation patterns**: Identify the booking characteristics and customer behaviors that are most associated with booking cancellations.
- **Build a predictive machine learning model**: Develop and evaluate a model that can identify customers who are likely to cancel their reservations before check-in.
- **Support proactive decision-makin**g: Enable hotel teams to take preventive actions, such as sending booking reminders, requiring deposits, or offering personalized incentives to reduce cancellations and improve room occupancy.

**Metric Evaluations**

**Type I Error (False Positive)**
- Prediction: The booking will be canceled.
- Actual: The booking is **not** canceled.

Consequence:

The hotel takes unnecessary actions, such as sending additional reminders, requesting a deposit, or offering incentives.
This increases operational costs and may negatively affect the customer experience.

**Type II Error (False Negative)**
- Prediction: The booking will not be canceled.
- Actual: The booking is canceled.

Consequence:

The hotel misses the opportunity to take preventive actions.
The room may remain vacant, resulting in lower occupancy and lost revenue.

---
## **Section 2. Data Understanding**
This section provides an overview of the dataset's structure, including its size and format, before performing data cleaning.

### **2.1 General Information**

In [ ]:
# Load Dataset
df = pd.read_csv('../data/raw/data_hotel_booking_demand.csv')
df.head()

,country,market_segment,previous_cancellations,booking_changes,deposit_type,days_in_waiting_list,customer_type,reserved_room_type,required_car_parking_spaces,total_of_special_requests,is_canceled
0,IRL,Offline TA/TO,0,0,No Deposit,0,Transient-Party,A,0,0,0
1,FRA,Online TA,0,0,No Deposit,0,Transient,A,0,2,0
2,PRT,Online TA,0,1,No Deposit,0,Transient,A,0,2,0
3,NLD,Online TA,0,0,No Deposit,0,Transient,A,0,1,1
4,PRT,Online TA,0,2,No Deposit,0,Transient,A,0,2,0


In [ ]:
# Dataset Dimension
print(f"Dataset Dimension: {df.shape}")

Dimensi Dataset: (83573, 11)


In [ ]:
# Count and Datatype for each Column
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 83573 entries, 0 to 83572
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   country                      83222 non-null  str  
 1   market_segment               83573 non-null  str  
 2   previous_cancellations       83573 non-null  int64
 3   booking_changes              83573 non-null  int64
 4   deposit_type                 83573 non-null  str  
 5   days_in_waiting_list         83573 non-null  int64
 6   customer_type                83573 non-null  str  
 7   reserved_room_type           83573 non-null  str  
 8   required_car_parking_spaces  83573 non-null  int64
 9   total_of_special_requests    83573 non-null  int64
 10  is_canceled                  83573 non-null  int64
dtypes: int64(6), str(5)
memory usage: 9.7 MB


### **2.2 Feature Information**

**Features Identification**

| Column | Type | Description |
| --- | --- | --- |
| country |string| Country of origin.|
| market_segment|string|Market segment designation. |
| previous_cancellations|integer| Number of previous bookings that were cancelled by the customer prior to the current booking|
| booking_changes|integer| Number of changes/amendments made to the booking from the moment the booking was entered on the PMS until the moment of check-in or cancellation|
| deposit_type|string| Indication on if the customer made a deposit to guarantee the booking.|
| days_in_waiting_list|integer| Number of days the booking was in the waiting list before it was confirmed to the customer|
| customer_type|string| Type of booking|
| reserved_room_type|string| Code of room type reserved. Code is presented instead of designation for anonymity reasons|
| required_car_parking_space|integer| Number of car parking spaces required by the customer|
| total_of_special_request|integer| Number of special requests made by the customer (e.g. twin bed or high floor)|
| is_canceled|integer| Value indicating if the booking was canceled (1) or not (0)|


In [39]:
# Cek nilai unik dan distribusi feature kategorikal
categorical_features = ['country', 'market_segment', 'deposit_type', 'customer_type', 'reserved_room_type']

for i in categorical_features:
    print(f"\n--- Kolom: {i} ---")
    print(f"Total Kategori Unique: {df[i].nunique()}")
    print("Distribusi Semua Kategori:")

    counts = df[i].value_counts(dropna=False)
    
    for i, (val, count) in enumerate(zip(counts.index, counts.values)):
        print(f" {i+1}. {val}: {count}")



--- Kolom: country ---
Total Kategori Unique: 162
Distribusi Semua Kategori:
 1. PRT: 34097
 2. GBR: 8495
 3. FRA: 7307
 4. ESP: 5996
 5. DEU: 5116
 6. ITA: 2658
 7. IRL: 2340
 8. BEL: 1648
 9. BRA: 1553
 10. USA: 1472
 11. NLD: 1433
 12. CHE: 1201
 13. CN: 886
 14. AUT: 873
 15. SWE: 724
 16. CHN: 709
 17. POL: 638
 18. ISR: 463
 19. RUS: 435
 20. NOR: 431
 21. nan: 351
 22. ROU: 341
 23. FIN: 316
 24. DNK: 308
 25. AUS: 301
 26. AGO: 243
 27. LUX: 181
 28. MAR: 179
 29. TUR: 163
 30. ARG: 154
 31. HUN: 138
 32. JPN: 129
 33. CZE: 117
 34. IND: 104
 35. KOR: 102
 36. GRC: 94
 37. SRB: 81
 38. DZA: 80
 39. HRV: 73
 40. IRN: 63
 41. ZAF: 60
 42. LTU: 59
 43. MEX: 57
 44. EST: 55
 45. BGR: 55
 46. NZL: 49
 47. COL: 47
 48. CHL: 43
 49. MOZ: 42
 50. UKR: 42
 51. SVN: 42
 52. ISL: 41
 53. SVK: 41
 54. THA: 40
 55. ARE: 38
 56. SAU: 37
 57. LVA: 36
 58. TWN: 34
 59. CYP: 33
 60. TUN: 31
 61. SGP: 27
 62. PHL: 27
 63. IDN: 27
 64. HKG: 26
 65. LBN: 24
 66. URY: 23
 67. NGA: 22
 68. EGY: 21


Pada country terlihat terdapat NaN (nomor 21) sebanyak 351 

In [41]:
# Cek nilai unik dan sebaran feature numerikal
numerical_features = ['previous_cancellations', 'booking_changes', 'days_in_waiting_list', 'required_car_parking_spaces', 'total_of_special_requests']

for col in numerical_features:
    print(f"\n--- Kolom: {col} ---")
    print(f"Total Nilai Unique: {df[col].nunique(dropna=False)}")
    print("Distribusi Semua Nilai:")
    
    counts = df[col].value_counts(dropna=False)
    
    for i, (val, count) in enumerate(zip(counts.index, counts.values)):
        print(f" {i+1}. Nilai {val}: {count} ")



--- Kolom: previous_cancellations ---
Total Nilai Unique: 15
Distribusi Semua Nilai:
 1. Nilai 0: 79060 
 2. Nilai 1: 4207 
 3. Nilai 2: 86 
 4. Nilai 3: 46 
 5. Nilai 24: 33 
 6. Nilai 11: 28 
 7. Nilai 6: 19 
 8. Nilai 4: 19 
 9. Nilai 26: 18 
 10. Nilai 25: 17 
 11. Nilai 19: 12 
 12. Nilai 13: 10 
 13. Nilai 14: 10 
 14. Nilai 5: 7 
 15. Nilai 21: 1 

--- Kolom: booking_changes ---
Total Nilai Unique: 19
Distribusi Semua Nilai:
 1. Nilai 0: 70873 
 2. Nilai 1: 8963 
 3. Nilai 2: 2652 
 4. Nilai 3: 639 
 5. Nilai 4: 260 
 6. Nilai 5: 90 
 7. Nilai 6: 39 
 8. Nilai 7: 23 
 9. Nilai 8: 10 
 10. Nilai 10: 5 
 11. Nilai 9: 4 
 12. Nilai 13: 4 
 13. Nilai 17: 2 
 14. Nilai 12: 2 
 15. Nilai 14: 2 
 16. Nilai 16: 2 
 17. Nilai 21: 1 
 18. Nilai 20: 1 
 19. Nilai 15: 1 

--- Kolom: days_in_waiting_list ---
Total Nilai Unique: 115
Distribusi Semua Nilai:
 1. Nilai 0: 80988 
 2. Nilai 39: 166 
 3. Nilai 58: 104 
 4. Nilai 31: 93 
 5. Nilai 44: 93 
 6. Nilai 46: 66 
 7. Nilai 35: 66 
 8. Nil

In [46]:
# Cek proporsi kelas target
target_counts = df['is_canceled'].value_counts()
target_percentage = df['is_canceled'].value_counts(normalize=True) * 100

for val, count, pct in zip(target_counts.index, target_counts.values, target_percentage.values):
    status = "Cancel" if val == 1 else "Check-in"
    print(f"[Class {val}] {status}: {count} data ({pct:.2f}%)")

[Class 0] Check-in: 52795 data (63.17%)
[Class 1] Cancel: 30778 data (36.83%)


### **2.3 Statistics Summary**

#### **2.3.1 Descriptive Statistics of Numerical Features**

In [42]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
previous_cancellations,83573.0,0.086798,0.841011,0.0,0.0,0.0,0.0,26.0
booking_changes,83573.0,0.220897,0.648635,0.0,0.0,0.0,0.0,21.0
days_in_waiting_list,83573.0,2.330561,17.673051,0.0,0.0,0.0,0.0,391.0
required_car_parking_spaces,83573.0,0.062999,0.246919,0.0,0.0,0.0,0.0,8.0
total_of_special_requests,83573.0,0.573211,0.795163,0.0,0.0,0.0,1.0,5.0
is_canceled,83573.0,0.368277,0.482340,0.0,0.0,0.0,1.0,1.0


#### **2.3.2 Descriptive Statistics of Categorical Features**

In [43]:
df.describe(include='object').T

/var/folders/f7/c5qn45190zdds3jh0t1vyfmh0000gn/T/ipykernel_3482/1760094569.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object').T


,count,unique,top,freq
country,83222,162,PRT,34097
market_segment,83573,8,Online TA,39460
deposit_type,83573,3,No Deposit,73352
customer_type,83573,4,Transient,62732
reserved_room_type,83573,10,A,60041


## **Section 3. Data Cleaning**

### **3.1 Missing Values**
Identify columns with missing values and determine the appropriate handling strategy, such as dropping, imputing, or retaining them based on business and analytical considerations.

In [ ]:
# check missing value
df.isnull().sum()

country                        351
market_segment                   0
previous_cancellations           0
booking_changes                  0
deposit_type                     0
days_in_waiting_list             0
customer_type                    0
reserved_room_type               0
required_car_parking_spaces      0
total_of_special_requests        0
is_canceled                      0
dtype: int64

The country column contains 351 missing values.

In [ ]:
# drop atau impute valuenya jadi 'unknown'

### **3.2 Duplicated Values**

In [ ]:
# check duplicated values
# Identify fully duplicated records across all columns

total_duplicates = df.duplicated().sum()
percentage_duplicates = (total_duplicates / len(df)) * 100

print(total_duplicates)
print(f"Percentage: {percentage_duplicates:.2f}%")

73371
Percentage: 87.79%


No duplicates were removed because the dataset has no unique transaction ID.

### **3.3 Data Consistency Check**
- Check for spelling errors or typos in categorical values.
- Check for inconsistent capitalization and formatting.
- Check for leading, trailing, or hidden whitespace characters.

In [ ]:
# Remove hidden whitespace from the beginning/end of strings and standardize text casing
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

print(f"Jumlah baris sebelum filter 'Undefined': {len(df)}")

df = df[df['market_segment'] != 'Undefined']
print(f"Jumlah baris setelah filter 'Undefined': {len(df)}")

Jumlah baris sebelum filter 'Undefined': 83572
Jumlah baris setelah filter 'Undefined': 83572


/var/folders/f7/c5qn45190zdds3jh0t1vyfmh0000gn/T/ipykernel_3482/3623355706.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns


### **3.4 Identify Anomaly Values**
- Check Distribution (Numerical Variable)
- Check Cardinality (Categorical Variable)

## **Section 4. Exploratory Data Analysis (EDA)**

### **4.1 Univariate Analysis**
- Distribusi target
- Distribusi fitur numerik
- Distribusi fitur kategorikal


### **4.2 Bivariate Analysis (terhadap Target)**

> 🎯 *Tujuan:* Mencari pola hubungan antara tiap fitur dengan target, untuk menjawab langsung Problem Statement di Section 1.2.

## **Section 5. Data Preparation**

### **5.1 Initialization**

In [ ]:
# Define feature and target


### **5.2 Constructing `Training` and `Testing` Data (from `Seen` Dataset)**

In [ ]:
# split into train and test

# reset index

# print x_train, y_train shape

### **5.3 Handling Imbalanced Data (jika relevan)**

> 🎯 *Tujuan:* Menangani ketimpangan proporsi kelas target supaya model tidak bias ke kelas mayoritas.

> 📌 Cek proporsi kelas target di Section 5.1. Kalau timpang (misal 90:10), pertimbangkan strategi seperti class_weight, SMOTE, atau undersampling — **tapi ingat, teknik resampling hanya boleh diterapkan pada data training**, tidak pernah pada data testing/unseen, supaya evaluasi tetap realistis.

### **5.4 Data Transformation (Feature Engineering)**

> 🎯 *Tujuan:* Melakukan encoding, scaling, atau transformasi lain agar data sesuai kebutuhan algoritma yang dipakai.

### **5.5 Feature Selection**


## **Section 6. Model Development**

### **6.1 Initialization**

### **6.2 Developing the Model Pipeline**

> 📌 Gunakan `Pipeline`/`ColumnTransformer` dari scikit-learn supaya seluruh langkah preprocessing (imputasi, encoding, scaling) ikut ter-*fit* hanya pada data training di setiap fold — ini mencegah data leakage antara fold CV.

### **6.3 Model Benchmarking (Comparing model base performance)**

### **6.4 Hyperparameter Tunning**


### **6.5 Best Model Evaluation**
- Evaluate model on data testing
- Confusion Matrix / Threshold Analysis (Classification) atau Residual Analysis (Regression)


### **6.6 Model Explanation and Interpretation**

#### **6.6.1 How model works**


#### **6.6.2 Model Explanation with SHAP**

## **Section 7. Model Deployment**


### **7.1 Export Model (joblib/pickle)**

### **7.2 Deployment Checklist**
- Versi library yang digunakan
- Format input yang diharapkan model
- Cara memuat ulang pipeline
.

## **Section 8. Conclusion and Recommendation**

### **8.1 Conclusion**
- Conclusion (Model)
- Conclusion (Business)


### **8.2 Recommendation**
- Recommendation (Model)
- Recommendation (Business)